# Equation (1)

The analysis uses the R&D firm-year handoff in `data/reg.dta`. Each rank is formed independently within fiscal year over the R&D sample after the paper's main-sample screens. The regression uses observations complete on XRET and all three rank variables. The handoff retains numeric fiscal-year label `fyear`, which the replay diagnostic uses to separate the original and added periods within the actual estimation sample.


In [ ]:
version 18
set more off

global dep "xret"
global rhs "gdelta grda garnoa"

use "data/reg.dta", clear
egen __miss = rowmiss(xret gdelta grda garnoa)
keep if __miss == 0

est clear
regress $dep $rhs
local pooled_n = e(N)
local pooled_r2 = e(r2)
local b_gdelta = _b[gdelta]
local se_gdelta = _se[gdelta]
quietly count if e(sample) & fyear >= 1979 & fyear <= 1998
local original_n = r(N)
quietly count if e(sample) & fyear >= 1999 & fyear <= 2003
local added_n = r(N)
quietly summarize fyear if e(sample) & fyear >= 1999 & fyear <= 2003, meanonly
local last_added = r(max)
display "EXT_SAMPLE s1 `original_n' `added_n' `last_added'"
estadd scalar original_n = `original_n'
estadd scalar added_n = `added_n'
estadd scalar last_added = `last_added'
est store eq1

esttab eq1 using "tables/eq1.md", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab eq1 using "tables/eq1.rtf", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab eq1 using "tables/eq1.csv", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab eq1 using "tables/eq1.tex", cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) r2 ar2 replace label
esttab eq1 using "artifacts/s1_aggregate.csv", cells(b(fmt(8)) se(par fmt(8))) stats(original_n added_n last_added N r2 ar2, labels("Original-period N" "Added-period N" "Last added fyear" "Pooled observations" "R-squared" "Adjusted R-squared")) replace label

estpost tabstat $dep $rhs, stat(N mean sd p25 p50 p75) c(s)
esttab using "tables/eq1_descriptive.md", cells("count(fmt(%9.0f)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") label replace
esttab using "tables/eq1_descriptive.rtf", cells("count(fmt(%9.0f)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") label replace
esttab using "tables/eq1_descriptive.csv", cells("count(fmt(%9.0f)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") label replace
esttab using "tables/eq1_descriptive.tex", cells("count(fmt(%9.0f)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") label replace
est clear
